# FOCUS-3D nuclei detector + physical PP submission

Detection is replaced by the pretrained **FOCUS-3D** 3D instance-segmentation
model (`model_final_nuclei.pth`, MaskFormer-3D, ~1.1B params). Everything else
(physical Hungarian relinking, short-track filtering, line fit, conservative
division recovery via `PHYSICAL_PP="full"`) is unchanged from the validated
0.910 notebook.

FOCUS-3D inference settings (validated locally on a V100, budgeted for T4):

- full-resolution frames `(64, 256, 256)`, one `infer_volume` call per frame
- `cell_radius=15` → skips XY resampling (native cell radius ~10-12 px is
  close enough to the model reference radius of 15 px)
- non-overlapping `stride=[32, 96, 96]` (= patch size) → ~4x fewer patches
- `batch_size=12`, fp32 weights + internal AMP autocast (peak ~6.7 GiB)
- per-frame adaptive `background_threshold = median(frame)`
- `z_ratio` derived from each dataset's physical scale (1.625 / 0.40625 = 4)
- the bundled runtime includes a **vectorized `masks_to_boxes_3d` patch**
  (bit-exact, replaces a 3600-iteration Python loop with GPU syncs that
  dominated CPU time): 5.9 s -> 3.6 s per frame on a V100, identical output
- optional `FOCUS3D_COMPILE=True` adds torch.compile (warm 2.9 s/frame on
  V100, but ~3-10 min one-time compile per worker; enable only when the
  hidden test set is large enough to amortize it)

Instance centroids become tracking nodes. No learned candidate edges are
emitted: the physical PP rebuilds links from coordinates alone
(`learned_edge_bonus=0` in the `full` preset), so dropping the edge
transformer loses nothing.

Local end-to-end validation (train video `44b6_12dfb391`, 100 frames, V100):

- 66,351 raw detections -> 60,625 nodes / 58,478 edges / 227 divisions after PP
  (estimated truth size: 58,672 nodes -> detection rate ~103%, no over-segmentation)
- sparse-truth check (788 annotated nodes): node recall 98.5%, annotated-edge
  recall 95.2%
- 3.6 s/frame (patched runtime), 6.73 GiB peak VRAM -> est. ~7-9 s/frame on T4,
  ~12-15 min/video; 2xT4 runs two videos concurrently -> 4 public videos in
  ~25-30 min

Required attached datasets:

- `qiweiyin/focus3d-nuclei-runtime` — FOCUS-3D runtime code + config +
  `models/model_final_nuclei.pth` (4.5 GB)
- `pilkwang/biohub-tracking-support-pack-50ep-v1` — offline wheels +
  `biohub_tracking` repo (io / graph / metrics)
- competition data `biohub-cell-tracking-during-development`


In [ ]:
try:
    import zarr, geff, tracksdata
    
except: 
    import os
    import sys
    import subprocess
    import importlib
    import importlib.util
    from pathlib import Path
    
    # Polars wheel in the support package uses the 32-bit runtime package.
    os.environ.setdefault("POLARS_PREFER_PKG", "32")
    
    SUPPORT_DIR = Path(
        "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"
    )
    WHEELS_DIR = SUPPORT_DIR / "wheels"
    
    if not WHEELS_DIR.exists():
        # Fallback for Kaggle's shorter mounted dataset path.
        candidates = list(Path("/kaggle/input").glob("**/wheels"))
        if not candidates:
            raise FileNotFoundError("Could not find the attached offline wheels directory")
        WHEELS_DIR = candidates[0]
    
    print("Offline wheels:", WHEELS_DIR)
    
    
    # These are packages supplied by the attached dataset.
    # Deliberately exclude:
    #   numpy, scipy, torch, pandas, dask, xarray, numba, llvmlite
    #
    # Kaggle already supplies compatible versions of those packages.
    OFFLINE_PACKAGES = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "imagecodecs==2026.6.26",
    ]
    
    
    def module_missing(module_name: str) -> bool:
        return importlib.util.find_spec(module_name) is None
    
    
    # Import name differs from pip package name in several cases.
    REQUIRED_IMPORTS = {
        "tracksdata": "tracksdata",
        "zarr": "zarr",
        "numcodecs": "numcodecs",
        "geff": "geff",
        "pyscipopt": "pyscipopt",
        "ilpy": "ilpy",
        "rustworkx": "rustworkx",
        "polars": "polars",
        "imagecodecs": "imagecodecs",
    }
    
    
    def purge_modules(module_roots):
        """
        Remove already-imported package modules from sys.modules.
    
        Normally this cell runs before imports, but this also protects against
        accidental imports performed by earlier Kaggle initialization code.
        """
        for root in module_roots:
            for name in list(sys.modules):
                if name == root or name.startswith(root + "."):
                    sys.modules.pop(name, None)
    
    
    def install_offline_packages():
        cmd = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-index",
            "--no-deps",
            "--find-links",
            str(WHEELS_DIR),
            *OFFLINE_PACKAGES,
        ]
    
        print("Installing attached packages without modifying NumPy/SciPy/Torch...")
        result = subprocess.run(
            cmd,
            text=True,
            capture_output=True,
        )
    
        if result.returncode != 0:
            print(result.stdout[-4000:])
            print(result.stderr[-4000:])
            raise RuntimeError("Offline dependency installation failed")
    
        purge_modules(REQUIRED_IMPORTS.values())
    
    
    install_offline_packages()
    
    
    # Verify the complete import chain needed by biohub_tracking.io.
    failures = {}
    
    for name, module_name in {
        **REQUIRED_IMPORTS,
        "numpy": "numpy",
        "scipy": "scipy",
        "dask": "dask.array",
        "xarray": "xarray",
    }.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    
    if failures:
        raise ImportError(
            "Dependency verification failed:\n"
            + "\n".join(f"{name}: {error}" for name, error in failures.items())
        )

import zarr, geff, tracksdata
print("All offline dependencies imported successfully.")


In [ ]:
# FOCUS-3D nuclei + physical PP. MODE=submit globs competition test/
# (replaced by hidden test at scoring time).

import sys
import shutil
from pathlib import Path

SUPPORT_CANDIDATES = [
    Path("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"),
    Path("/kaggle/input/pilkwang/biohub-tracking-support-pack-50ep-v1"),
    Path("/kaggle/input/biohub-tracking-support-pack-50ep-v1"),
]
SUPPORT_DIR = next((p for p in SUPPORT_CANDIDATES if (p / "repo/src").exists()), None)
if SUPPORT_DIR is None:
    hits = list(Path("/kaggle/input").glob("**/repo/src"))
    if not hits:
        raise FileNotFoundError("Could not find biohub-tracking-support-pack repo/src")
    SUPPORT_DIR = hits[0].parent.parent
sys.path.append(str(SUPPORT_DIR / "repo/src"))

from biohub_tracking.io import open_dataset, save_graph

import os
import contextlib
import tempfile
import zarr
import numpy as np
from tqdm import tqdm
import json
import glob
import csv
import pandas as pd
from joblib import Parallel, delayed

import torch

import tracksdata as td
import polars as pl

from geff import GeffMetadata
from biohub_tracking.metrics import (
    evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)

#--------------------------------------------
MODE = "submit"  # "submit" | "local"

KAGGLE_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development"
if MODE == "local":
    valid_id = ["44b6_0113de3b", "44b6_0b24845f", "6bba_05b6850b", "6bba_05db0fb1"]
    valid_dir = f"{KAGGLE_DIR}/train"
elif MODE == "submit":
    glob_file = glob.glob(f"{KAGGLE_DIR}/test/*.zarr")
    valid_id = sorted([f.split("/")[-1][:-5] for f in glob_file])
    valid_dir = f"{KAGGLE_DIR}/test"
else:
    raise ValueError(MODE)


# ---- FOCUS-3D runtime + nuclei checkpoint resolution --------------------

def _ok_focus3d_root(root: Path) -> bool:
    return (
        (root / "focus3d_runtime" / "focus3d").is_dir()
        and (root / "configs" / "3d_test.yaml").is_file()
        and (
            (root / "models" / "model_final_nuclei.pth").is_file()
            or (root / "models " / "model_final_nuclei.pth").is_file()
        )
    )


def _resolve_focus3d_root() -> Path | None:
    candidates = [
        Path("/kaggle/input/datasets/qiweiyin/focus3d-nuclei-runtime"),
        Path("/kaggle/input/qiweiyin/focus3d-nuclei-runtime"),
        Path("/kaggle/input/focus3d-nuclei-runtime"),
        Path("/home/qiwei/FOCUS-3D"),  # local dev machine
    ]
    for p in candidates:
        if p.exists() and _ok_focus3d_root(p):
            return p
    for hit in Path("/kaggle/input").glob("**/model_final_nuclei.pth"):
        root = hit.parent.parent
        if _ok_focus3d_root(root):
            return root
    return None


FOCUS3D_ROOT = _resolve_focus3d_root()
if FOCUS3D_ROOT is None:
    raise FileNotFoundError(
        "FOCUS-3D runtime/weights not found. Attach a dataset containing "
        "focus3d_runtime/, configs/3d_test.yaml and models/model_final_nuclei.pth "
        "(e.g. qiweiyin/focus3d-nuclei-runtime)."
    )

FOCUS3D_CONFIG = FOCUS3D_ROOT / "configs" / "3d_test.yaml"
_models_dir = FOCUS3D_ROOT / "models"
if not (_models_dir / "model_final_nuclei.pth").is_file():
    _models_dir = FOCUS3D_ROOT / "models "  # local dir keeps a trailing space
FOCUS3D_WEIGHTS = _models_dir / "model_final_nuclei.pth"

runtime_path = str(FOCUS3D_ROOT / "focus3d_runtime")
if runtime_path not in sys.path:
    sys.path.insert(0, runtime_path)

print("MODE:", MODE)
print("SUPPORT_DIR:", SUPPORT_DIR)
print("FOCUS3D_ROOT:", FOCUS3D_ROOT)
print("FOCUS3D_WEIGHTS:", FOCUS3D_WEIGHTS)
print("valid_id:", len(valid_id), valid_id[:8])
print("setup ok!!!!!")


In [ ]:
# FOCUS-3D detection module (replaces the TemporalUNet detector + edge transformer)

DEVICE = "cuda"
USE_MULTI_GPU = True
USE_ILP = False          # physical PP replaces learned edges / ILP
USE_TTA = False          # FOCUS-3D runs single-view (no flip TTA); kept for the eval cell
PHYSICAL_PP = "full"     # control: "motion_gap_short_linefit"

# FOCUS-3D inference knobs (locally validated: 4.8 s/frame, 6.7 GiB on V100;
# ~95% center agreement with the slower overlapping-stride baseline)
FOCUS3D_CELL_RADIUS = 15.0      # == model reference radius -> no XY resampling
FOCUS3D_STRIDE = [32, 96, 96]   # == patch size -> non-overlapping windows
FOCUS3D_BATCH_SIZE = 12
FOCUS3D_SCORE_THRESH = 0.6
FOCUS3D_MASK_THRESH = 0.5
FOCUS3D_MIN_EDGE_AREA = 64
FOCUS3D_TOPK = 300
FOCUS3D_COMPILE = True  # torch.compile: warm ~2.9 s/frame vs 3.6 s, but costs
                         # ~3-10 min one-time compile per worker. Enable only
                         # if the hidden test set is large (>~8 videos).

# Pipeline: prefetch next zarr while GPU runs; overlap physical PP with next video.
PREFETCH_VOLUME = True
OVERLAP_PP = True
SHOW_FRAME_TQDM = False

# CUDA speed knobs
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cuda.matmul.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

from scipy import ndimage as ndi
import tifffile


@contextlib.contextmanager
def suppress_output():
    """Context manager to suppress stdout and stderr."""
    with open(os.devnull, "w") as devnull:
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            yield


# io helper ----------------------------------

def load_volume_full(sample_id):
    """Full-resolution raw volume (T,64,256,256) uint16 + physical scale.

    FOCUS-3D performs its own 1-99 percentile normalization inside
    infer_volume, so no manual normalization here.
    """
    zarr_file = f"{valid_dir}/{sample_id}.zarr"
    ds = open_dataset(zarr_file, normalize=False, load_image=False, require_tracks=False)
    zarr_arr = zarr.open_group(str(ds.zarr_path), mode="r")["0"]
    volume = zarr_arr[:]
    meta = {"scale_um": tuple(float(s) for s in ds.scale)}
    return volume, meta


# Graph building (unchanged contract: nodes carry t/z/y/x, edges edge_prob/edge_dist)
def build_graph(coord, edge):
    graph = td.graph.InMemoryGraph()
    for key in ["z", "y", "x"]:
        graph.add_node_attr_key(key, pl.Float64, -999999.0)

    node_ids = graph.bulk_add_nodes([
        {"t": int(t), "z": float(z), "y": float(y), "x": float(x)}
        for t, z, y, x in coord
    ])

    if edge:
        graph.add_edge_attr_key("edge_prob", pl.Float64, 0.0)
        graph.add_edge_attr_key("edge_dist", pl.Float64, 0.0)
        graph.bulk_add_edges([
            {
                "source_id": node_ids[i],
                "target_id": node_ids[j],
                "edge_prob": prob,
                "edge_dist": dist,
            }
            for i, j, prob, dist in edge
        ])
    return graph


# FOCUS-3D per-frame detection ----------------------------------------------

def _frame_centroids(instance_map: np.ndarray) -> np.ndarray:
    """(N, 3) float zyx centroids of all instances in one frame."""
    ids = np.unique(instance_map[instance_map > 0])
    if ids.size == 0:
        return np.zeros((0, 3), dtype=np.float32)
    coms = ndi.center_of_mass(instance_map > 0, instance_map, ids)
    return np.asarray(coms, dtype=np.float32)


def predict_one_focus3d(model, volume, meta, device, work_root):
    """Run FOCUS-3D on every frame; return nodes and an empty edge list.

    Nodes are full-resolution voxel centroids [t, z, y, x]. Candidate edges
    are intentionally empty: the physical PP (`motion_relink`) rebuilds all
    links from coordinates and the `full` preset sets `learned_edge_bonus=0`,
    so learned edges would be ignored anyway.
    """
    from focus3d.segmentation.FOCUS3D.inference_win import infer_volume

    T = int(volume.shape[0])
    z_ratio = float(meta["scale_um"][0]) / float(meta["scale_um"][1])
    out_node = []
    out_edge = []  # no learned candidate edges

    tmp_dir = Path(tempfile.mkdtemp(prefix="focus3d_frames_", dir=str(work_root)))
    n_failed = 0
    try:
        for t in tqdm(range(T), total=T, leave=False, disable=not SHOW_FRAME_TQDM):
            frame = np.ascontiguousarray(volume[t])
            input_path = tmp_dir / f"f{t:03d}.tif"
            out_dir = tmp_dir / f"o{t:03d}"
            tifffile.imwrite(
                str(input_path), frame,
                metadata={"axes": "ZYX"}, photometric="minisblack",
            )
            try:
                with suppress_output():
                    result = infer_volume(
                        image_path=input_path,
                        config_file=FOCUS3D_CONFIG,
                        weights_path=FOCUS3D_WEIGHTS,
                        model=model,
                        device=str(device),
                        output_dir=out_dir,
                        z_ratio=z_ratio,
                        lower_percentile=1.0,
                        upper_percentile=99.0,
                        data_loader_num_workers=0,
                        cell_radius=FOCUS3D_CELL_RADIUS,
                        background_threshold=float(np.median(frame)),
                        stride=FOCUS3D_STRIDE,
                        batch_size=FOCUS3D_BATCH_SIZE,
                        score_thresh=FOCUS3D_SCORE_THRESH,
                        mask_thresh=FOCUS3D_MASK_THRESH,
                        min_edge_area=FOCUS3D_MIN_EDGE_AREA,
                        topk_postprocess=FOCUS3D_TOPK,
                        save_intermediate=False,
                    )
                centroids = _frame_centroids(np.asarray(result["instance_map"]))
            except Exception as exc:
                n_failed += 1
                print(f"  [focus3d] frame {t} failed: {type(exc).__name__}: {exc}", flush=True)
                centroids = np.zeros((0, 3), dtype=np.float32)
            finally:
                input_path.unlink(missing_ok=True)
                shutil.rmtree(out_dir, ignore_errors=True)

            if centroids.size:
                out_node.extend(
                    [[t, float(z), float(y), float(x)] for z, y, x in centroids]
                )
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    print(
        f"focus3d: frames={T} nodes={len(out_node)} failed_frames={n_failed}",
        flush=True,
    )
    return out_node, out_edge


print(
    f"FOCUS-3D detection: cell_radius={FOCUS3D_CELL_RADIUS} "
    f"stride={FOCUS3D_STRIDE} bs={FOCUS3D_BATCH_SIZE} "
    f"score_thr={FOCUS3D_SCORE_THRESH} | PHYSICAL_PP={PHYSICAL_PP} "
    f"PREFETCH={PREFETCH_VOLUME} OVERLAP_PP={OVERLAP_PP}"
)
print("modeling ok !!!")


In [ ]:
# Self-contained physical linker/PP copied from the locally tested repository.
"""Metric-aligned physical post-processing for point-based cell tracks.

The implementation is intentionally independent from model training.  It takes
the node coordinates and candidate edges emitted by inference, then applies a
small set of deterministic operations inspired by the public 0.917 solution:
physical Hungarian relinking, snap-only one-frame gap repair, short-component
filtering, line-fit smoothing, and conservative division recovery.
"""

from __future__ import annotations

from dataclasses import dataclass, replace
from typing import Iterable

import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree


EdgeTuple = tuple[int, int, float, float]
_BIG = 1e12


@dataclass(frozen=True)
class PhysicalPostprocessConfig:
    scale_um: tuple[float, float, float] = (1.625, 0.40625, 0.40625)
    enforce_next_frame: bool = True
    edge_max_um: float = 14.0

    motion_relink: bool = False
    motion_tight_um: float = 6.0
    motion_relaxed_um: float = 10.0
    velocity_weight: float = 0.5
    confidence_weight: float = 2.0
    learned_edge_bonus: float = 0.0

    gap_close: bool = False
    gap_gate_um: float = 9.0
    gap_snap_um: float = 3.2
    gap_min_score: float | None = None
    gap_cap_fraction: float = 0.003

    filter_short: bool = False
    min_track_length: int = 6
    keep_division_components: bool = True

    linefit: bool = False
    linefit_weight: float = 0.8
    linefit_window: int = 2

    safe_divisions: bool = False
    division_parent_um: float = 12.0
    division_sister_um: float = 15.0
    division_existing_child_um: float = 10.0
    division_diverge_um: float = 2.25
    division_sister_weight: float = 0.15
    division_frame_cap_fraction: float = 0.0076
    division_global_cap_fraction: float = 0.00375

    prune_isolated: bool = False


def preset_config(name: str) -> PhysicalPostprocessConfig:
    """Return a cumulative ablation preset.

    ``full`` follows the active operation order in the public 0.917 notebook.
    Its gap closer is deliberately stricter in this repository: it can only
    reuse an isolated detection already present in the input graph.
    """

    base = PhysicalPostprocessConfig()
    presets = {
        "raw": replace(
            base,
            enforce_next_frame=False,
            edge_max_um=0.0,
        ),
        "motion": replace(base, motion_relink=True),
        "motion_learned": replace(
            base,
            motion_relink=True,
            learned_edge_bonus=0.75,
        ),
        "motion_gap": replace(
            base,
            motion_relink=True,
            gap_close=True,
        ),
        "motion_gap_short": replace(
            base,
            motion_relink=True,
            gap_close=True,
            filter_short=True,
        ),
        "motion_gap_short_linefit": replace(
            base,
            motion_relink=True,
            gap_close=True,
            filter_short=True,
            linefit=True,
        ),
        "full": replace(
            base,
            motion_relink=True,
            gap_close=True,
            filter_short=True,
            linefit=True,
            safe_divisions=True,
            prune_isolated=True,
        ),
    }
    try:
        return presets[name]
    except KeyError as exc:
        raise ValueError(
            f"Unknown preset {name!r}; choose from {', '.join(presets)}"
        ) from exc


def available_presets() -> tuple[str, ...]:
    return (
        "raw",
        "motion",
        "motion_learned",
        "motion_gap",
        "motion_gap_short",
        "motion_gap_short_linefit",
        "full",
    )


def _physical_positions(
    coords: np.ndarray,
    node_ids: Iterable[int],
    scale_um: np.ndarray,
) -> np.ndarray:
    ids = np.asarray(list(node_ids), dtype=np.int64)
    if ids.size == 0:
        return np.empty((0, 3), dtype=np.float64)
    return coords[ids, 1:].astype(np.float64) * scale_um[None, :]


def _distance_um(
    coords: np.ndarray,
    source: int,
    target: int,
    scale_um: np.ndarray,
) -> float:
    delta = (coords[source, 1:] - coords[target, 1:]) * scale_um
    return float(np.linalg.norm(delta))


def _logit(values: np.ndarray) -> np.ndarray:
    clipped = np.clip(values, 1e-4, 1.0 - 1e-4)
    return np.log(clipped / (1.0 - clipped))


def _clean_edges(
    coords: np.ndarray,
    raw_edges: list[dict[str, float | int]],
    cfg: PhysicalPostprocessConfig,
    stats: dict[str, int | float],
) -> list[dict[str, float | int]]:
    scale_um = np.asarray(cfg.scale_um, dtype=np.float64)
    cleaned: list[dict[str, float | int]] = []
    for edge in raw_edges:
        source = int(edge["source_id"])
        target = int(edge["target_id"])
        if source < 0 or target < 0 or source >= len(coords) or target >= len(coords):
            stats["dropped_missing_node_edges"] += 1
            continue
        dt = int(round(float(coords[target, 0]) - float(coords[source, 0])))
        if cfg.enforce_next_frame and dt != 1:
            stats["dropped_nonconsecutive_edges"] += 1
            continue
        distance = _distance_um(coords, source, target, scale_um)
        if cfg.edge_max_um > 0 and distance > cfg.edge_max_um:
            stats["dropped_long_edges"] += 1
            continue
        item = dict(edge)
        item["distance_um"] = distance
        cleaned.append(item)
    return cleaned


def _motion_relink(
    coords: np.ndarray,
    node_ids: set[int],
    raw_edges: list[dict[str, float | int]],
    cfg: PhysicalPostprocessConfig,
    node_scores: np.ndarray | None,
    stats: dict[str, int | float],
) -> list[dict[str, float | int]]:
    scale_um = np.asarray(cfg.scale_um, dtype=np.float64)
    ids_by_t: dict[int, list[int]] = {}
    for node_id in sorted(node_ids):
        ids_by_t.setdefault(int(round(float(coords[node_id, 0]))), []).append(node_id)

    learned_probs: dict[tuple[int, int], float] = {}
    for edge in raw_edges:
        key = (int(edge["source_id"]), int(edge["target_id"]))
        prob = float(edge.get("edge_prob", 0.0))
        if np.isfinite(prob):
            learned_probs[key] = max(learned_probs.get(key, -np.inf), prob)

    selected: list[dict[str, float | int]] = []
    velocity_by_node: dict[int, np.ndarray] = {}
    all_times = sorted(ids_by_t)
    for t in all_times:
        source_ids = ids_by_t.get(t, [])
        target_ids = ids_by_t.get(t + 1, [])
        if not source_ids or not target_ids:
            continue

        source_pos = _physical_positions(coords, source_ids, scale_um)
        target_pos = _physical_positions(coords, target_ids, scale_um)
        previous_velocity = np.stack(
            [velocity_by_node.get(node_id, np.zeros(3)) for node_id in source_ids]
        )
        predicted = source_pos + cfg.velocity_weight * previous_velocity
        raw_distance = np.linalg.norm(
            source_pos[:, None, :] - target_pos[None, :, :], axis=2
        )
        motion_distance = np.linalg.norm(
            predicted[:, None, :] - target_pos[None, :, :], axis=2
        )
        cost = motion_distance.copy()

        if node_scores is not None and cfg.confidence_weight > 0:
            source_score = node_scores[np.asarray(source_ids)]
            target_score = node_scores[np.asarray(target_ids)]
            valid = np.isfinite(source_score)[:, None] & np.isfinite(target_score)[None, :]
            similarity = np.abs(
                _logit(np.nan_to_num(source_score, nan=0.5))[:, None]
                - _logit(np.nan_to_num(target_score, nan=0.5))[None, :]
            )
            cost += np.where(valid, cfg.confidence_weight * similarity, 0.0)

        if cfg.learned_edge_bonus > 0:
            bonus = np.zeros_like(cost)
            for i, source in enumerate(source_ids):
                for j, target in enumerate(target_ids):
                    bonus[i, j] = learned_probs.get((source, target), 0.0)
            cost -= cfg.learned_edge_bonus * bonus

        unmatched_source = list(range(len(source_ids)))
        unmatched_target = list(range(len(target_ids)))
        frame_matches: list[tuple[int, int, str]] = []
        for pass_name, gate in (
            ("tight", cfg.motion_tight_um),
            ("relaxed", cfg.motion_relaxed_um),
        ):
            if not unmatched_source or not unmatched_target:
                break
            sub_raw = raw_distance[np.ix_(unmatched_source, unmatched_target)]
            sub_cost = cost[np.ix_(unmatched_source, unmatched_target)]
            gated = np.where(sub_raw <= gate, sub_cost, _BIG)
            rows, cols = linear_sum_assignment(gated)
            accepted: list[tuple[int, int]] = []
            for row, col in zip(rows, cols):
                if gated[row, col] >= _BIG:
                    continue
                source_index = unmatched_source[int(row)]
                target_index = unmatched_target[int(col)]
                accepted.append((source_index, target_index))
                frame_matches.append((source_index, target_index, pass_name))
            if accepted:
                used_source = {source for source, _ in accepted}
                used_target = {target for _, target in accepted}
                unmatched_source = [i for i in unmatched_source if i not in used_source]
                unmatched_target = [i for i in unmatched_target if i not in used_target]

        for source_index, target_index, pass_name in frame_matches:
            source = source_ids[source_index]
            target = target_ids[target_index]
            probability = learned_probs.get((source, target), 0.0)
            selected.append(
                {
                    "source_id": source,
                    "target_id": target,
                    "edge_prob": probability,
                    "distance_um": float(raw_distance[source_index, target_index]),
                    "motion_distance_um": float(
                        motion_distance[source_index, target_index]
                    ),
                    "motion_pass": pass_name,
                }
            )
            velocity_by_node[target] = target_pos[target_index] - source_pos[source_index]
            stats[f"motion_{pass_name}_edges"] += 1
        stats["motion_frames"] += 1

    stats["motion_edges"] = len(selected)
    return selected


def _edge_maps(
    edges: list[dict[str, float | int]],
) -> tuple[dict[int, list[int]], dict[int, list[int]]]:
    outgoing: dict[int, list[int]] = {}
    incoming: dict[int, list[int]] = {}
    for edge in edges:
        source = int(edge["source_id"])
        target = int(edge["target_id"])
        outgoing.setdefault(source, []).append(target)
        incoming.setdefault(target, []).append(source)
    return outgoing, incoming


def _gap_close(
    coords: np.ndarray,
    node_ids: set[int],
    edges: list[dict[str, float | int]],
    cfg: PhysicalPostprocessConfig,
    node_scores: np.ndarray | None,
    stats: dict[str, int | float],
) -> list[dict[str, float | int]]:
    if not cfg.gap_close or not edges:
        return edges

    scale_um = np.asarray(cfg.scale_um, dtype=np.float64)
    outgoing, incoming = _edge_maps(edges)
    ids_by_t: dict[int, list[int]] = {}
    for node_id in sorted(node_ids):
        ids_by_t.setdefault(int(round(float(coords[node_id, 0]))), []).append(node_id)

    max_gaps = max(1, int(cfg.gap_cap_fraction * max(len(node_ids), 1)))
    used_candidates: set[int] = set()
    added: list[dict[str, float | int]] = []
    max_t = max(ids_by_t, default=-1)
    for t in range(max_t - 1):
        if len(added) // 2 >= max_gaps:
            break
        ends = [
            node_id
            for node_id in ids_by_t.get(t, [])
            if node_id not in outgoing and len(incoming.get(node_id, [])) == 1
        ]
        starts = [
            node_id
            for node_id in ids_by_t.get(t + 2, [])
            if node_id not in incoming and len(outgoing.get(node_id, [])) == 1
        ]
        candidates = [
            node_id
            for node_id in ids_by_t.get(t + 1, [])
            if node_id not in outgoing
            and node_id not in incoming
            and node_id not in used_candidates
            and (
                cfg.gap_min_score is None
                or node_scores is None
                or (
                    np.isfinite(node_scores[node_id])
                    and node_scores[node_id] >= cfg.gap_min_score
                )
            )
        ]
        if not ends or not starts or not candidates:
            continue

        end_pos = _physical_positions(coords, ends, scale_um)
        start_pos = _physical_positions(coords, starts, scale_um)
        distance = np.linalg.norm(end_pos[:, None, :] - start_pos[None, :, :], axis=2)
        gated = np.where(distance <= cfg.gap_gate_um, distance, _BIG)
        rows, cols = linear_sum_assignment(gated)
        proposals = sorted(
            (
                float(distance[row, col]),
                ends[int(row)],
                starts[int(col)],
            )
            for row, col in zip(rows, cols)
            if gated[row, col] < _BIG
        )
        candidate_pos = _physical_positions(coords, candidates, scale_um)
        tree = cKDTree(candidate_pos)
        stats["gap_endpoint_pairs"] += len(proposals)
        for _, end, start in proposals:
            if len(added) // 2 >= max_gaps:
                break
            midpoint = 0.5 * (
                coords[end, 1:].astype(np.float64)
                + coords[start, 1:].astype(np.float64)
            )
            distance_to_midpoint, candidate_index = tree.query(midpoint * scale_um)
            candidate = candidates[int(candidate_index)]
            if distance_to_midpoint > cfg.gap_snap_um or candidate in used_candidates:
                continue
            used_candidates.add(candidate)
            added.extend(
                [
                    {
                        "source_id": end,
                        "target_id": candidate,
                        "edge_prob": 0.0,
                        "distance_um": _distance_um(coords, end, candidate, scale_um),
                        "gap_edge": 1,
                    },
                    {
                        "source_id": candidate,
                        "target_id": start,
                        "edge_prob": 0.0,
                        "distance_um": _distance_um(coords, candidate, start, scale_um),
                        "gap_edge": 1,
                    },
                ]
            )
            outgoing[end] = [candidate]
            incoming[candidate] = [end]
            outgoing[candidate] = [start]
            incoming[start] = [candidate]
            stats["gap_nodes_reused"] += 1

    stats["gap_edges_added"] = len(added)
    return [*edges, *added]


def _components(
    node_ids: set[int],
    edges: list[dict[str, float | int]],
) -> list[list[int]]:
    parent = {node_id: node_id for node_id in node_ids}

    def find(node_id: int) -> int:
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    def union(a: int, b: int) -> None:
        if a not in parent or b not in parent:
            return
        root_a, root_b = find(a), find(b)
        if root_a != root_b:
            parent[root_a] = root_b

    for edge in edges:
        union(int(edge["source_id"]), int(edge["target_id"]))
    grouped: dict[int, list[int]] = {}
    for node_id in sorted(node_ids):
        grouped.setdefault(find(node_id), []).append(node_id)
    return list(grouped.values())


def _filter_short(
    node_ids: set[int],
    edges: list[dict[str, float | int]],
    cfg: PhysicalPostprocessConfig,
    stats: dict[str, int | float],
) -> tuple[set[int], list[dict[str, float | int]]]:
    if not cfg.filter_short or cfg.min_track_length <= 1:
        return node_ids, edges
    outgoing, _ = _edge_maps(edges)
    keep: set[int] = set()
    removed_components = 0
    for component in _components(node_ids, edges):
        has_division = any(len(outgoing.get(node_id, [])) >= 2 for node_id in component)
        if len(component) >= cfg.min_track_length or (
            cfg.keep_division_components and has_division
        ):
            keep.update(component)
        else:
            removed_components += 1
    filtered_edges = [
        edge
        for edge in edges
        if int(edge["source_id"]) in keep and int(edge["target_id"]) in keep
    ]
    stats["short_components_removed"] = removed_components
    stats["short_nodes_removed"] = len(node_ids) - len(keep)
    stats["short_edges_removed"] = len(edges) - len(filtered_edges)
    return keep, filtered_edges


def _linefit(
    coords: np.ndarray,
    node_ids: set[int],
    edges: list[dict[str, float | int]],
    cfg: PhysicalPostprocessConfig,
    stats: dict[str, int | float],
) -> None:
    if not cfg.linefit or cfg.linefit_weight <= 0:
        return
    outgoing, incoming = _edge_maps(edges)
    original = {node_id: coords[node_id, 1:].astype(np.float64).copy() for node_id in node_ids}
    updates: dict[int, np.ndarray] = {}
    for node_id in sorted(node_ids):
        neighbourhood = [(0, node_id)]
        current = node_id
        for step in range(1, cfg.linefit_window + 1):
            predecessors = incoming.get(current, [])
            if len(predecessors) != 1:
                break
            current = predecessors[0]
            neighbourhood.append((-step, current))
        current = node_id
        for step in range(1, cfg.linefit_window + 1):
            successors = outgoing.get(current, [])
            if len(successors) != 1:
                break
            current = successors[0]
            neighbourhood.append((step, current))
        if len(neighbourhood) < 3:
            continue
        dt = np.asarray([delta for delta, _ in neighbourhood], dtype=np.float64)
        xyz = np.stack([original[other] for _, other in neighbourhood])
        fitted = np.asarray(
            [np.polyval(np.polyfit(dt, xyz[:, axis], 1), 0.0) for axis in range(3)]
        )
        if np.isfinite(fitted).all():
            updates[node_id] = (
                (1.0 - cfg.linefit_weight) * original[node_id]
                + cfg.linefit_weight * fitted
            )
    for node_id, xyz in updates.items():
        coords[node_id, 1:] = xyz
    stats["linefit_nodes"] = len(updates)


def _safe_divisions(
    coords: np.ndarray,
    node_ids: set[int],
    edges: list[dict[str, float | int]],
    cfg: PhysicalPostprocessConfig,
    stats: dict[str, int | float],
) -> list[dict[str, float | int]]:
    if not cfg.safe_divisions or not edges:
        return edges
    scale_um = np.asarray(cfg.scale_um, dtype=np.float64)
    outgoing, incoming = _edge_maps(edges)
    ids_by_t: dict[int, list[int]] = {}
    for node_id in sorted(node_ids):
        ids_by_t.setdefault(int(round(float(coords[node_id, 0]))), []).append(node_id)

    orphan_ids: dict[int, list[int]] = {}
    orphan_trees: dict[int, cKDTree] = {}
    for t, ids in ids_by_t.items():
        orphans = [node_id for node_id in ids if node_id not in incoming]
        orphan_ids[t] = orphans
        if orphans:
            orphan_trees[t] = cKDTree(_physical_positions(coords, orphans, scale_um))

    proposals: list[tuple[float, int, int, int]] = []
    for parent in sorted(node_ids):
        children = outgoing.get(parent, [])
        if len(children) != 1 or len(incoming.get(parent, [])) != 1:
            continue
        child = children[0]
        parent_t = int(round(float(coords[parent, 0])))
        if int(round(float(coords[child, 0]))) != parent_t + 1:
            continue
        if _distance_um(coords, parent, child, scale_um) > cfg.division_existing_child_um:
            continue
        tree = orphan_trees.get(parent_t + 1)
        if tree is None:
            continue
        parent_pos = coords[parent, 1:].astype(np.float64) * scale_um
        child_pos = coords[child, 1:].astype(np.float64) * scale_um
        nearest_distance, nearest_index = tree.query(child_pos)
        nearest_orphan = orphan_ids[parent_t + 1][int(nearest_index)]
        for orphan_index in tree.query_ball_point(parent_pos, cfg.division_parent_um):
            candidate = orphan_ids[parent_t + 1][int(orphan_index)]
            if candidate == child or candidate != nearest_orphan:
                continue
            parent_distance = _distance_um(coords, parent, candidate, scale_um)
            sister_distance = _distance_um(coords, child, candidate, scale_um)
            if nearest_distance > cfg.division_sister_um or sister_distance > cfg.division_sister_um:
                continue
            child_next = outgoing.get(child, [])
            candidate_next = outgoing.get(candidate, [])
            if len(child_next) != 1 or len(candidate_next) != 1:
                continue
            next_sister_distance = _distance_um(
                coords, child_next[0], candidate_next[0], scale_um
            )
            if next_sister_distance - sister_distance < cfg.division_diverge_um:
                continue
            score = parent_distance + cfg.division_sister_weight * sister_distance
            proposals.append((score, parent_t, parent, candidate))

    proposals.sort()
    used_parents: set[int] = set()
    used_candidates: set[int] = set(incoming)
    per_frame: dict[int, int] = {}
    global_cap = max(1, int(cfg.division_global_cap_fraction * max(len(node_ids), 1)))
    added: list[dict[str, float | int]] = []
    for _, parent_t, parent, candidate in proposals:
        if len(added) >= global_cap:
            break
        if parent in used_parents or candidate in used_candidates:
            continue
        frame_cap = max(
            1,
            int(cfg.division_frame_cap_fraction * len(ids_by_t.get(parent_t, []))),
        )
        if per_frame.get(parent_t, 0) >= frame_cap:
            continue
        added.append(
            {
                "source_id": parent,
                "target_id": candidate,
                "edge_prob": 0.0,
                "distance_um": _distance_um(coords, parent, candidate, scale_um),
                "safe_division": 1,
            }
        )
        used_parents.add(parent)
        used_candidates.add(candidate)
        per_frame[parent_t] = per_frame.get(parent_t, 0) + 1

    stats["division_candidates"] = len(proposals)
    stats["divisions_added"] = len(added)
    return [*edges, *added]


def _prune_isolated(
    node_ids: set[int],
    edges: list[dict[str, float | int]],
    stats: dict[str, int | float],
) -> set[int]:
    incident = {
        int(edge[key])
        for edge in edges
        for key in ("source_id", "target_id")
    }
    kept = node_ids & incident
    stats["isolated_nodes_removed"] = len(node_ids) - len(kept)
    return kept


def _graph_counts(
    node_ids: set[int],
    edges: list[dict[str, float | int]],
) -> tuple[int, int]:
    components = len(_components(node_ids, edges)) if node_ids else 0
    outgoing, _ = _edge_maps(edges)
    divisions = sum(len(children) >= 2 for children in outgoing.values())
    return components, divisions


def _validate(
    coords: np.ndarray,
    node_ids: set[int],
    edges: list[dict[str, float | int]],
) -> None:
    seen: set[tuple[int, int]] = set()
    incoming_count: dict[int, int] = {}
    for edge in edges:
        source = int(edge["source_id"])
        target = int(edge["target_id"])
        if source == target:
            raise ValueError("Post-processing created a self edge")
        if source not in node_ids or target not in node_ids:
            raise ValueError("Post-processing edge references a removed node")
        pair = (source, target)
        if pair in seen:
            raise ValueError(f"Post-processing created duplicate edge {pair}")
        seen.add(pair)
        if float(coords[target, 0]) <= float(coords[source, 0]):
            raise ValueError("Post-processing created a non-forward edge")
        incoming_count[target] = incoming_count.get(target, 0) + 1
        if incoming_count[target] > 1:
            raise ValueError(f"Node {target} has more than one parent")


def postprocess_tracks(
    coords: np.ndarray,
    raw_edges: Iterable[EdgeTuple],
    config: PhysicalPostprocessConfig,
    node_scores: np.ndarray | None = None,
) -> tuple[np.ndarray, list[EdgeTuple], dict[str, int | float]]:
    """Post-process one movie and return reindexed coordinates and edges."""

    coords = np.asarray(coords, dtype=np.float64).copy()
    if coords.ndim != 2 or coords.shape[1] != 4:
        raise ValueError(f"coords must have shape (N, 4), got {coords.shape}")
    if node_scores is not None:
        node_scores = np.asarray(node_scores, dtype=np.float64)
        if node_scores.shape != (len(coords),):
            raise ValueError("node_scores must have one value per coordinate")

    edge_records = [
        {
            "source_id": int(source),
            "target_id": int(target),
            "edge_prob": float(probability),
            "edge_dist": float(distance),
        }
        for source, target, probability, distance in raw_edges
    ]
    stats: dict[str, int | float] = {
        "raw_nodes": len(coords),
        "raw_edges": len(edge_records),
        "dropped_missing_node_edges": 0,
        "dropped_nonconsecutive_edges": 0,
        "dropped_long_edges": 0,
        "motion_frames": 0,
        "motion_tight_edges": 0,
        "motion_relaxed_edges": 0,
        "motion_edges": 0,
        "gap_endpoint_pairs": 0,
        "gap_nodes_reused": 0,
        "gap_edges_added": 0,
        "short_components_removed": 0,
        "short_nodes_removed": 0,
        "short_edges_removed": 0,
        "linefit_nodes": 0,
        "division_candidates": 0,
        "divisions_added": 0,
        "isolated_nodes_removed": 0,
    }
    node_ids = set(range(len(coords)))
    edges = _clean_edges(coords, edge_records, config, stats)
    if config.motion_relink:
        edges = _motion_relink(coords, node_ids, edges, config, node_scores, stats)
    edges = _gap_close(coords, node_ids, edges, config, node_scores, stats)
    node_ids, edges = _filter_short(node_ids, edges, config, stats)
    _linefit(coords, node_ids, edges, config, stats)
    edges = _safe_divisions(coords, node_ids, edges, config, stats)
    if config.prune_isolated:
        node_ids = _prune_isolated(node_ids, edges, stats)
        edges = [
            edge
            for edge in edges
            if int(edge["source_id"]) in node_ids
            and int(edge["target_id"]) in node_ids
        ]

    _validate(coords, node_ids, edges)
    old_ids = sorted(node_ids)
    remap = {old_id: new_id for new_id, old_id in enumerate(old_ids)}
    output_coords = coords[np.asarray(old_ids, dtype=np.int64)].astype(np.float32)
    output_edges: list[EdgeTuple] = []
    for edge in edges:
        source = int(edge["source_id"])
        target = int(edge["target_id"])
        probability = float(edge.get("edge_prob", 0.0))
        voxel_distance = float(
            np.linalg.norm(coords[source, 1:] - coords[target, 1:])
        )
        output_edges.append(
            (remap[source], remap[target], probability, voxel_distance)
        )

    components, divisions = _graph_counts(node_ids, edges)
    stats["final_nodes"] = len(output_coords)
    stats["final_edges"] = len(output_edges)
    stats["final_components"] = components
    stats["final_divisions"] = divisions
    return output_coords, output_edges, stats


print("physical PP loaded:", available_presets())


In [ ]:
######## FOCUS-3D nuclei inference + physical PP ########

predict_dir = "/kaggle/working/my_predict"
os.makedirs(predict_dir, exist_ok=True)


def build_focus3d_model(device: torch.device):
    """Build the FOCUS-3D MaskFormer once per worker on the given GPU."""
    runtime_path = str(FOCUS3D_ROOT / "focus3d_runtime")
    if runtime_path not in sys.path:
        sys.path.insert(0, runtime_path)
    from focus3d.segmentation.FOCUS3D.inference_win import (
        build_predictor,
        setup_cfg,
    )
    cfg = setup_cfg(str(FOCUS3D_CONFIG), str(FOCUS3D_WEIGHTS), device=str(device))
    model = build_predictor(cfg)
    if FOCUS3D_COMPILE:
        model = torch.compile(model, dynamic=True)
        print(f"focus3d model torch.compile enabled on {device}", flush=True)
    print(f"focus3d model loaded on {device}", flush=True)
    return model


def _physical_pp_and_save(out_node, out_edge, meta, sample_id, gpu_id, t_video0):
    """CPU-side deterministic PP + GEFF write, overlapped with GPU inference."""
    import time as _time
    coords = np.asarray(out_node, dtype=np.float32)
    if coords.size == 0:
        coords = np.zeros((0, 4), dtype=np.float32)
    cfg = replace(
        preset_config(PHYSICAL_PP),
        scale_um=tuple(float(s) for s in meta["scale_um"]),
    )
    coords, edges, stats = postprocess_tracks(coords, out_edge, cfg)
    graph = build_graph(coords, edges)
    save_graph(graph, f"{predict_dir}/{sample_id}.geff")
    print(
        f"[GPU {gpu_id}] {sample_id}: PP={PHYSICAL_PP} "
        f"nodes={graph.num_nodes()}, edges={graph.num_edges()}, "
        f"divisions={stats['final_divisions']} "
        f"({_time.perf_counter() - t_video0:.1f}s)",
        flush=True,
    )
    return sample_id


def run_worker(gpu_id: int, subset_id):
    torch.cuda.set_device(gpu_id)
    device = torch.device(f"cuda:{gpu_id}")
    model = build_focus3d_model(device)

    from concurrent.futures import ThreadPoolExecutor
    import time as _time

    work_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

    subset_id = list(subset_id)
    # 2 threads: volume prefetch + physical PP/save can overlap with GPU predict.
    pool = ThreadPoolExecutor(max_workers=2) if (PREFETCH_VOLUME or OVERLAP_PP) else None
    load_fut = None
    pp_fut = None

    try:
        if pool is not None and PREFETCH_VOLUME and subset_id:
            load_fut = pool.submit(load_volume_full, subset_id[0])

        for i, sample_id in enumerate(subset_id):
            t_video0 = _time.perf_counter()
            if load_fut is not None:
                volume, meta = load_fut.result()
                load_fut = None
            else:
                volume, meta = load_volume_full(sample_id)

            if pool is not None and PREFETCH_VOLUME and i + 1 < len(subset_id):
                load_fut = pool.submit(load_volume_full, subset_id[i + 1])

            out_node, out_edge = predict_one_focus3d(
                model, volume, meta, device, work_root
            )
            del volume

            if pp_fut is not None:
                pp_fut.result()
                pp_fut = None

            save_args = (out_node, out_edge, meta, sample_id, gpu_id, t_video0)
            if pool is not None and OVERLAP_PP:
                pp_fut = pool.submit(_physical_pp_and_save, *save_args)
            else:
                _physical_pp_and_save(*save_args)

        if pp_fut is not None:
            pp_fut.result()
        if load_fut is not None:
            load_fut.result()
    finally:
        if pool is not None:
            pool.shutdown(wait=True)

    del model
    torch.cuda.empty_cache()
    return gpu_id


if USE_MULTI_GPU and torch.cuda.device_count() >= 2:
    subset_id0 = valid_id[0::2]
    subset_id1 = valid_id[1::2]
    result = Parallel(n_jobs=2, backend="loky", verbose=10)(
        [
            delayed(run_worker)(0, subset_id0),
            delayed(run_worker)(1, subset_id1),
        ]
    )
    print(result)
else:
    print(
        f"single-GPU mode: USE_MULTI_GPU={USE_MULTI_GPU}, "
        f"device_count={torch.cuda.device_count()}",
        flush=True,
    )
    run_worker(0, valid_id)

print("predict ok !!!")


In [ ]:
from pathlib import Path
#submission

SUBMISSION_PATH = "submission.csv"
SUBMISSION_COLUMN = [
    "id",
    "dataset",
    "row_type",
    "node_id",
    "t",
    "z",
    "y",
    "x",
    "source_id",
    "target_id",
]
 

glob_file = glob.glob(f"{predict_dir}/*.geff")
print(f"predict_dir: {len(glob_file)}")

if len(glob_file) != len(valid_id):
    raise RuntimeError(
        f"Expected {len(valid_id)} GEFF files, found {len(glob_file)}"
    )

row_id = 0
total_num_node = 0
total_num_edge = 0

with Path(SUBMISSION_PATH).open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=SUBMISSION_COLUMN)
    writer.writeheader()

    for sample_id in valid_id:
        dataset = sample_id
        graph = td.graph.IndexedRXGraph.from_geff(f"{predict_dir}/{sample_id}.geff")[0]

        node_row = list(graph.node_attrs().iter_rows(named=True))
        edge_row = list(graph.edge_attrs().iter_rows(named=True))

        node_id = {int(row["node_id"]) for row in node_row}
        if not node_id:
            raise AssertionError(f"{dataset}: physical-PP graph contains no nodes")

        # Direct node export. Rounding is only CSV serialization.
        for row in sorted(node_row, key=lambda x: int(x["node_id"])):
            writer.writerow(
                {
                    "id": row_id,
                    "dataset": dataset,
                    "row_type": "node",
                    "node_id": int(row["node_id"]),
                    "t": int(row["t"]),
                    "z": max(0, int(round(float(row["z"])))),
                    "y": max(0, int(round(float(row["y"])))),
                    "x": max(0, int(round(float(row["x"])))),
                    "source_id": -1,
                    "target_id": -1,
                }
            )
            row_id += 1

         
        for row in edge_row:
            source_id = int(row["source_id"])
            target_id = int(row["target_id"])
 
            if source_id not in node_id or target_id not in node_id:
                raise AssertionError(
                    f"{dataset}: dangling ILP edge {source_id}->{target_id}"
                )

            writer.writerow(
                {
                    "id": row_id,
                    "dataset": dataset,
                    "row_type": "edge",
                    "node_id": -1,
                    "t": -1,
                    "z": -1,
                    "y": -1,
                    "x": -1,
                    "source_id": source_id,
                    "target_id": target_id,
                }
            )
            row_id += 1
             
        total_num_node += len(node_row)
        total_num_edge += len(edge_row)

    
submit_df = pd.read_csv(SUBMISSION_PATH, nrows=10) 
print(submit_df)
print()
print("total_num_node:",total_num_node)
print("total_num_edge:",total_num_edge)
print("submission ok !!!")


In [ ]:
#evalute
if MODE=="local":
    
    metric_df =[]
    for sample_id in valid_id:
        truth_file = f"{KAGGLE_DIR}/train/{sample_id}.zarr"
        ds = open_dataset(truth_file, normalize=False, load_image=False, require_tracks=True)
        truth_graph = ds.tracks
        #print(truth_graph)
    
        predict_file = f"{predict_dir}/{sample_id}.geff"
        pred_result = td.graph.IndexedRXGraph.from_geff(predict_file)
        pred_graph = pred_result[0]# if isinstance(pred_result, tuple) else pred_result
        #print(pred_graph)
    
        print(sample_id, "---------------------------")
        er = evaluate(
            pred_graph,
            truth_graph,
            scale=ds.scale,
            max_distance=7.0,
        )
        print("edge TP:", er.edge_tp)
        print("edge FP:", er.edge_fp)
        print("edge FN:", er.edge_fn)
        print("division TP:", er.division_tp)
        print("division FP:", er.division_fp)
        print("division FN:", er.division_fn)
    
        recall = node_recall(pred_graph, truth_graph)
        print("node_recall:", recall)
    
        meta = GeffMetadata.read(truth_file.replace(".zarr",".geff"))
        #print("meta:", meta)
        n_total = float(meta.extra["estimated_number_of_nodes"])
    
        metrics = per_sample_metrics(
            er=er,
            n_total=n_total,
            node_recall=recall,
        )
        metric_df.append(metrics)
        print("n_total:", n_total)
        print("metrics:", metrics)
        
    print() 
    metric_df = pd.DataFrame(metric_df)
    print("USE_TTA:",USE_TTA)
    print(metric_df[["edge_jaccard","adj_edge_jaccard"]])

print("evaluation ok!!!")
